#  PhoGPT ABSA Batch Labeling (Multiple Files)

**Process 11 files tự động** - Upload ZIP folder và process tất cả

---

## ️ Setup

1. **Zip your 11 Excel files** vào 1 file (ví dụ: `data_files.zip`)
2. **Runtime Settings**: Menu → Runtime → Change runtime type → **GPU (T4)**
3. **Upload ZIP**: Chạy cell "Upload ZIP File"
4. **Run all cells**: Runtime → Run all
5. **Download results**: Tất cả files labeled sẽ được zip lại và download

---

##  Step 1: Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes pandas openpyxl tqdm

import torch
print(f" GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

##  Step 2: Upload ZIP File (Chứa 11 Excel Files)

In [ ]:
from google.colab import files
import zipfile
import os
import glob

print("Upload your ZIP file containing all Excel files...")
uploaded = files.upload()

# Extract ZIP
zip_filename = list(uploaded.keys())[0]
print(f"\nExtracting {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('input_data')

# Find all Excel files
excel_files = glob.glob('input_data/**/*.xlsx', recursive=True)
excel_files = [f for f in excel_files if not f.startswith('input_data/__MACOSX')]

print(f"\n Found {len(excel_files)} Excel files:")
for f in excel_files:
    print(f"   - {os.path.basename(f)}")

## ️ Step 3: Configuration

In [ ]:
# ========== CONFIGURATION ==========

# Limit per file (None = process all reviews in each file)
LIMIT_PER_FILE = None  # Set to number (e.g. 100) for testing

# Use 8-bit quantization
USE_8BIT = True

# Model settings
MODEL_NAME = "vinai/PhoGPT-7B5-Instruct"
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.1

# Output folder
OUTPUT_FOLDER = "phogpt_labeled_output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Files to process: {len(excel_files)}")
print(f"Limit per file: {LIMIT_PER_FILE if LIMIT_PER_FILE else 'All reviews'}")
print(f"8-bit mode: {USE_8BIT}")
print(f"Output folder: {OUTPUT_FOLDER}")

##  Step 4: Load PhoGPT Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import json
import re
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Loading PhoGPT model: {MODEL_NAME}...")
print(f"This may take 5-10 minutes...\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

load_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
}

if USE_8BIT:
    load_kwargs["load_in_8bit"] = True

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)

if not USE_8BIT:
    model = model.to(DEVICE)

model.eval()
print(" Model loaded!")

## ️ Step 5: Define Labeling Functions

In [ ]:
ASPECTS = [
    'Chất lượng sản phẩm',
    'Hiệu năng & Trải nghiệm',
    'Đúng mô tả',
    'Giá cả & Khuyến mãi',
    'Vận chuyển',
    'Đóng gói',
    'Dịch vụ & Thái độ Shop',
    'Bảo hành & Đổi trả',
    'Tính xác thực',
]

def create_prompt(review: str) -> str:
    prompt = f"""### Instruction:
Bạn là chuyên gia phân tích cảm xúc cho bình luận thương mại điện tử.
Nhiệm vụ: Phân tích bình luận sau theo 9 khía cạnh và gán nhãn cảm xúc.

**9 Khía cạnh:**
1. Chất lượng sản phẩm - chất liệu, độ bền
2. Hiệu năng & Trải nghiệm - trải nghiệm sử dụng
3. Đúng mô tả - so với hình ảnh/mô tả
4. Giá cả & Khuyến mãi - giá trị, ưu đãi
5. Vận chuyển - tốc độ giao hàng
6. Đóng gói - bao bì
7. Dịch vụ & Thái độ Shop - CSKH
8. Bảo hành & Đổi trả - chính sách
9. Tính xác thực - hàng thật/giả

**Nhãn:** 1 (tích cực), 0 (trung lập), -1 (tiêu cực), 2 (không đề cập), [-1,1] (đa cực)

**Bình luận:** \"{review}\"

Trả về JSON:
### Response:
```json
{{
  \"Chất lượng sản phẩm\": <nhãn>,
  \"Hiệu năng & Trải nghiệm\": <nhãn>,
  \"Đúng mô tả\": <nhãn>,
  \"Giá cả & Khuyến mãi\": <nhãn>,
  \"Vận chuyển\": <nhãn>,
  \"Đóng gói\": <nhãn>,
  \"Dịch vụ & Thái độ Shop\": <nhãn>,
  \"Bảo hành & Đổi trả\": <nhãn>,
  \"Tính xác thực\": <nhãn>
}}
```"""
    return prompt

def parse_response(response: str) -> dict:
    try:
        json_match = re.search(r'\{[^}]+\}', response, re.DOTALL)
        if json_match:
            labels = json.loads(json_match.group(0))
            return {asp: labels.get(asp, 2) for asp in ASPECTS}
    except:
        pass
    return {asp: 2 for asp in ASPECTS}

def label_review(review: str) -> dict:
    if not review or not str(review).strip():
        return {asp: 2 for asp in ASPECTS}
    
    prompt = create_prompt(str(review))
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    
    if torch.cuda.is_available():
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]
    return parse_response(response)

print(" Functions defined")

##  Step 6: Process All Files (BATCH)

In [ ]:
import time

print(f"\n{'='*60}")
print(f"BATCH PROCESSING {len(excel_files)} FILES")
print(f"{'='*60}\n")

total_reviews = 0
start_time = time.time()

for file_idx, input_file in enumerate(excel_files, 1):
    filename = os.path.basename(input_file)
    
    print(f"\n[{file_idx}/{len(excel_files)}] Processing: {filename}")
    print(f"{'-'*60}")
    
    # Load file
    df = pd.read_excel(input_file)
    original_count = len(df)
    
    # Apply limit
    if LIMIT_PER_FILE:
        df = df.head(LIMIT_PER_FILE)
    
    print(f"Reviews: {len(df)} (total in file: {original_count})")
    
    # Initialize columns
    for aspect in ASPECTS:
        df[aspect] = 2
    
    # Label each review
    for idx in tqdm(range(len(df)), desc=f"File {file_idx}"):
        review = df.iloc[idx]['reviewContent']
        try:
            labels = label_review(review)
            for aspect, label in labels.items():
                df.at[idx, aspect] = label
        except Exception as e:
            continue
    
    # Save output
    output_file = os.path.join(OUTPUT_FOLDER, f"labeled_{filename}")
    df.to_excel(output_file, index=False)
    
    total_reviews += len(df)
    elapsed = time.time() - start_time
    avg_time = elapsed / total_reviews if total_reviews > 0 else 0
    
    print(f" Saved: {output_file}")
    print(f"Progress: {total_reviews} reviews | Avg: {avg_time:.1f}s/review")

print(f"\n{'='*60}")
print(f" BATCH COMPLETE!")
print(f"Total reviews processed: {total_reviews}")
print(f"Total time: {(time.time() - start_time) / 60:.1f} minutes")
print(f"{'='*60}")

##  Step 7: Summary Statistics

In [ ]:
# Load all output files and show combined stats
output_files = glob.glob(f"{OUTPUT_FOLDER}/*.xlsx")

print(f"\n{'='*60}")
print(f"COMBINED STATISTICS ({len(output_files)} files)")
print(f"{'='*60}\n")

all_dfs = [pd.read_excel(f) for f in output_files]
combined_df = pd.concat(all_dfs, ignore_index=True)

for aspect in ASPECTS:
    mentions = (combined_df[aspect] != 2).sum()
    pos = (combined_df[aspect] == 1).sum()
    neg = (combined_df[aspect] == -1).sum()
    neu = (combined_df[aspect] == 0).sum()
    multi = combined_df[aspect].astype(str).str.contains('\[').sum()
    
    print(f"{aspect:30} | Mentions: {mentions:5} | POS: {pos:4} | NEG: {neg:4} | NEU: {neu:4} | Multi: {multi:3}")

print(f"\n{'='*60}")

##  Step 8: Download Results (ZIP All Files)

In [ ]:
import shutil

# Create ZIP of all output files
output_zip = "phogpt_labeled_results.zip"
shutil.make_archive(output_zip.replace('.zip', ''), 'zip', OUTPUT_FOLDER)

print(f" Created ZIP: {output_zip}")
print(f"   Contains {len(output_files)} labeled files\n")

# Download
from google.colab import files
print("Downloading ZIP file...")
files.download(output_zip)

print("\n DONE! Extract ZIP to get all 11 labeled files.")

---

##  Notes

### Workflow Summary:
1. Upload ZIP chứa 11 Excel files
2. Model tự động process từng file
3. Kết quả được lưu riêng cho mỗi file
4. Tất cả được zip lại và download

### Time Estimates:
- **~1000 reviews/file × 11 files = ~11,000 reviews**
- **Estimated: 30-50 hours** (yes, very long!)
- **T4 GPU limit: 12 hours/session** → Need multiple sessions

### Recommendations:
1. **Test với LIMIT_PER_FILE = 100 trước**
2. **Process batch nhỏ** (2-3 files/session)
3. **Hoặc dùng Colab Pro** (longer sessions)
4. **Hoặc dùng Gemini API** (faster!)

---